In [21]:

from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import *
from delta.tables import DeltaTable


spark = (
    SparkSession.builder
    .appName("MinIO-PostgreSQL")
    .master("local[*]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.jars", "/home/jovyan/postgresql-42.7.1.jar")  # PostgreSQL JDBC driver
    .getOrCreate()
)

hconf = spark.sparkContext._jsc.hadoopConfiguration()
hconf.set("fs.s3a.endpoint", "http://minio:9000")
hconf.set("fs.s3a.access.key", "matrix")
hconf.set("fs.s3a.secret.key", "matrix123")
hconf.set("fs.s3a.path.style.access", "true")
hconf.set("fs.s3a.connection.ssl.enabled", "false")
hconf.set("fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")

In [3]:
bronze="s3a://lazimsiz/bronze"
bronze_delta="s3a://lazimsiz/bronze_delta"


In [4]:
import os
fp='/home/jovyan/work'
os.listdir(fp)
##containerdeki fayllari gormek ucun

['files', 'test_scripts']

In [5]:
df_card=spark.read.csv('/home/jovyan/work/files/card.csv',header=True,inferSchema=True)
df_customer=spark.read.csv('/home/jovyan/work/files/customer.csv',header=True,inferSchema=True)
df_trs=spark.read.csv('/home/jovyan/work/files/trs.csv',header=True,inferSchema=True)

In [6]:
import pandas as pd
from IPython.display import FileLink
df = pd.read_csv('/home/jovyan/work/files/trs.csv')
filename = 'trs_json.json'
df.to_json(filename, orient='records')
display(FileLink(filename))

/home/jovyan/trs_json.json

In [7]:
df_json_spark_test=spark.read.format('json').load('/home/jovyan/work/test_scripts/trs_json.json')

In [8]:
df_card.write.mode("overwrite").option("header","true").csv(f"{bronze}/card.csv")
df_trs.write.mode("overwrite").option("header","true").csv(f"{bronze}/trs.csv")
df_customer.write.mode("overwrite").option("header","true").csv(f"{bronze}/customer.csv")
##burda adi fayl kimi bronze buckete yukleyirem

In [9]:
df_card.write.mode("overwrite").format("delta").save(f"{bronze_delta}/card")

In [10]:
df_card.write.mode("overwrite").format("delta").save(f"{bronze_delta}/card")
df_trs.write.mode("overwrite").format("delta").save(f"{bronze_delta}/trs")
df_customer.write.mode("overwrite").format("delta").save(f"{bronze_delta}/customer")
##burda delta kimi yazdim


In [13]:
df_card.show()

+-------+-----------+----------------+---------+-------+
|card_id|customer_id|     card_number|card_type| status|
+-------+-----------+----------------+---------+-------+
|   K001|       C001|4111111111111111|    DEBIT| ACTIVE|
|   K002|       C002|5222222222222222|   CREDIT| ACTIVE|
|   K003|       C003|6333333333333333|    DEBIT|BLOCKED|
|   K004|       C004|7444444444444444|   CREDIT| ACTIVE|
|   K005|       C005|8555555555555555|    DEBIT| ACTIVE|
+-------+-----------+----------------+---------+-------+



In [ ]:
#schemani yoxla
df_card.printSchema()


root
 |-- card_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- card_number: long (nullable = true)
 |-- card_type: string (nullable = true)
 |-- status: string (nullable = true)



In [18]:
#explicit schema yaratmaq
df_card_schema=StructType(
    [
        
        StructField("card_id",StringType(),True),
        StructField("customer_id",StringType(),True),
        StructField("card_number",StringType(),True),
        StructField("card_type",StringType(),True),
        StructField("status",StringType(),True),
     
    ]

)

In [ ]:
##schemani movcud datasete tetbiq et
##burda containerin icinden oxuyur
df_card=spark.read.csv(
    '/home/jovyan/work/files/card.csv',
    schema=df_card_schema,
    header=True)

In [20]:
df_card.printSchema()

root
 |-- card_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- card_number: string (nullable = true)
 |-- card_type: string (nullable = true)
 |-- status: string (nullable = true)

